# 🌾 AgriFlow AI - Production Deployment

## Intelligent Cascaded Machine Learning Pipeline for Precision Agriculture

### Objective

This notebook demonstrates the deployment version of AgriFlow AI.

Unlike the Model Development notebook, which focuses on experimentation and model evaluation, this notebook focuses on the production workflow by integrating the best-performing models from each stage into a unified agricultural decision support system.

### Pipeline

Farmer Input
↓
Crop Recommendation
↓
Yield Prediction
↓
Fertilizer Recommendation
↓
Final Farmer Report

In [2]:
import warnings
warnings.filterwarnings("ignore")

import pandas as pd
import numpy as np

from sklearn.model_selection import train_test_split

from sklearn.compose import ColumnTransformer

from sklearn.pipeline import Pipeline

from sklearn.preprocessing import OneHotEncoder

from sklearn.ensemble import (
    RandomForestClassifier,
    RandomForestRegressor
)

In [3]:
crop_df = pd.read_csv(r"D:\AgriFlow-AI\data\Crop_recommendation.csv")

yield_df = pd.read_csv(r"D:\AgriFlow-AI\data\Crop Yeild Data.csv")

fert_df = pd.read_csv(r"D:\AgriFlow-AI\data\fertilizer_recommendation.csv")

In [4]:
print("Crop Dataset :", crop_df.shape)

print("Yield Dataset :", yield_df.shape)

print("Fertilizer Dataset :", fert_df.shape)

Crop Dataset : (2200, 8)
Yield Dataset : (19689, 13)
Fertilizer Dataset : (10000, 20)


# 🌱 Stage 1 – Crop Recommendation Pipeline

The Crop Recommendation model predicts the most suitable crop using soil nutrients and environmental conditions.

Unlike the experimentation notebook, this implementation uses a Scikit-learn Pipeline, making preprocessing and prediction consistent for new farmer inputs.

In [5]:
crop = crop_df.copy()

crop["npk_total"] = crop["N"] + crop["P"] + crop["K"]

X_crop = crop.drop("label", axis=1)

y_crop = crop["label"]

X_crop.head()

,N,P,K,temperature,humidity,ph,rainfall,npk_total
0,90,42,43,20.879744,82.002744,6.502985,202.935536,175
1,85,58,41,21.770462,80.319644,7.038096,226.655537,184
2,60,55,44,23.004459,82.320763,7.840207,263.964248,159
3,74,35,40,26.491096,80.158363,6.980401,242.864034,149
4,78,42,42,20.130175,81.604873,7.628473,262.717340,162


In [6]:
X_train_crop, X_test_crop, y_train_crop, y_test_crop = train_test_split(
    X_crop,
    y_crop,
    test_size=0.2,
    random_state=42,
    stratify=y_crop
)

In [7]:
crop_pipeline = Pipeline([
    (
        "model",
        RandomForestClassifier(
            n_estimators=200,
            random_state=42
        )
    )
])

crop_pipeline.fit(X_train_crop, y_train_crop)

,steps,"[('model', ...)]"
,transform_input,None
,memory,None
,verbose,False
,n_estimators,200
,criterion,'gini'
,max_depth,None
,min_samples_split,2
,min_samples_leaf,1
,min_weight_fraction_leaf,0.0
,max_features,'sqrt'


In [8]:
from sklearn.metrics import accuracy_score

crop_pred = crop_pipeline.predict(X_test_crop)

print("Crop Pipeline Accuracy :", accuracy_score(y_test_crop, crop_pred))

Crop Pipeline Accuracy : 0.9931818181818182


In [9]:
import joblib
import os

os.makedirs("models", exist_ok=True)

joblib.dump(crop_pipeline, "models/crop_pipeline.pkl")

print("✅ Crop Pipeline Saved Successfully")

✅ Crop Pipeline Saved Successfully


In [10]:
import os
import joblib

# Create the models folder if it doesn't exist
os.makedirs("models", exist_ok=True)

# Save the trained crop pipeline
joblib.dump(crop_pipeline, "models/crop_pipeline.pkl")

print("✅ Crop pipeline saved successfully!")

✅ Crop pipeline saved successfully!


In [11]:
import os

print(os.listdir("models"))

['crop_pipeline.pkl']


In [12]:
loaded_crop_pipeline = joblib.load("models/crop_pipeline.pkl")

print("Pipeline loaded successfully!")

Pipeline loaded successfully!


In [13]:
sample = X_test_crop.iloc[[0]]

prediction = loaded_crop_pipeline.predict(sample)

print("Predicted Crop :", prediction[0])
print("Actual Crop    :", y_test_crop.iloc[0])

Predicted Crop : orange
Actual Crop    : orange


In [14]:
import joblib

crop_pipeline = joblib.load("models/crop_pipeline.pkl")

yield_pipeline = joblib.load("models/yield_pipeline.pkl")

fert_pipeline = joblib.load("models/fert_pipeline.pkl")

print("✅ All models loaded successfully!")

✅ All models loaded successfully!


In [15]:
print(type(crop_pipeline))

print(type(yield_pipeline))

print(type(fert_pipeline))

<class 'sklearn.ensemble._forest.RandomForestClassifier'>
<class 'sklearn.ensemble._forest.RandomForestClassifier'>
<class 'sklearn.ensemble._forest.RandomForestClassifier'>


In [24]:
farmer_input = {
    "n": 90,
    "p": 42,
    "k": 43,
    "temperature": 26.5,
    "humidity": 82,
    "ph": 6.5,
    "rainfall": 202
}

In [25]:
farmer_df = pd.DataFrame([farmer_input])

farmer_df["npk_total"] = (
    farmer_df["n"] +
    farmer_df["p"] +
    farmer_df["k"]
)

recommended_crop = crop_pipeline.predict(farmer_df)[0]

print("Recommended Crop:", recommended_crop)

Recommended Crop: rice


In [26]:
print(yield_pipeline.feature_names_in_)

['ph' 'soil_moisture' 'organic_carbon' 'electrical_conductivity' 'n' 'p'
 'k' 'temperature' 'humidity' 'rainfall' 'fertilizer_used_last_season'
 'npk_total' 'temp_humidity' 'Predicted_Yield' 'soil_type_Loamy'
 'soil_type_Sandy' 'soil_type_Silt' 'crop_type_Maize' 'crop_type_Potato'
 'crop_type_Rice' 'crop_type_Sugarcane' 'crop_type_Tomato'
 'crop_type_Wheat' 'crop_growth_stage_Harvest' 'crop_growth_stage_Sowing'
 'crop_growth_stage_Vegetative' 'season_Rabi' 'season_Zaid'
 'irrigation_type_Drip' 'irrigation_type_Rainfed'
 'irrigation_type_Sprinkler' 'previous_crop_Maize' 'previous_crop_Potato'
 'previous_crop_Rice' 'previous_crop_Sugarcane' 'previous_crop_Tomato'
 'previous_crop_Wheat' 'region_East' 'region_North' 'region_South'
 'region_West']


In [27]:
print(len(yield_pipeline.feature_names_in_))

41


In [30]:
print("Crop Model:")
print(crop_pipeline.feature_names_in_)

Crop Model:
['n' 'p' 'k' 'temperature' 'humidity' 'ph' 'rainfall' 'npk_total']


In [29]:
print("Yield Model:")
print(yield_pipeline.feature_names_in_)

Yield Model:
['ph' 'soil_moisture' 'organic_carbon' 'electrical_conductivity' 'n' 'p'
 'k' 'temperature' 'humidity' 'rainfall' 'fertilizer_used_last_season'
 'npk_total' 'temp_humidity' 'Predicted_Yield' 'soil_type_Loamy'
 'soil_type_Sandy' 'soil_type_Silt' 'crop_type_Maize' 'crop_type_Potato'
 'crop_type_Rice' 'crop_type_Sugarcane' 'crop_type_Tomato'
 'crop_type_Wheat' 'crop_growth_stage_Harvest' 'crop_growth_stage_Sowing'
 'crop_growth_stage_Vegetative' 'season_Rabi' 'season_Zaid'
 'irrigation_type_Drip' 'irrigation_type_Rainfed'
 'irrigation_type_Sprinkler' 'previous_crop_Maize' 'previous_crop_Potato'
 'previous_crop_Rice' 'previous_crop_Sugarcane' 'previous_crop_Tomato'
 'previous_crop_Wheat' 'region_East' 'region_North' 'region_South'
 'region_West']


In [36]:
import joblib

model = joblib.load("models/yield_pipeline.pkl")

print(type(model))

<class 'sklearn.ensemble._forest.RandomForestClassifier'>


# 🌾 Stage 2 - Yield Prediction Deployment

In [38]:
yield_df = pd.read_csv(r"D:\AgriFlow-AI\data\Crop Yeild Data.csv")

In [39]:
yield_df.columns = yield_df.columns.str.lower()

yield_df["recommended_crop"] = yield_df["crop"]

yield_df.drop(columns=["crop"], inplace=True)

yield_df["temp_range"] = (
    yield_df["max_temperature"] -
    yield_df["min_temperature"]
)

In [40]:
yield_df.head()

,crop_year,season,state,area,production,annual_rainfall,fertilizer,pesticide,yield,avg_temperature,max_temperature,min_temperature,recommended_crop,temp_range
0,1997,Whole Year,Assam,73814.0,56708,2051.4,7024878.38,22882.34,0.796,23.692,33.435,14.779,Arecanut,18.656
1,1997,Kharif,Assam,6637.0,4685,2051.4,631643.29,2057.47,0.710,23.692,33.435,14.779,Arhar/Tur,18.656
2,1997,Kharif,Assam,796.0,22,2051.4,75755.32,246.76,0.238,23.692,33.435,14.779,Castor seed,18.656
3,1997,Whole Year,Assam,19656.0,126905000,2051.4,1870661.52,6093.36,5238.052,23.692,33.435,14.779,Coconut,18.656
4,1997,Kharif,Assam,1739.0,794,2051.4,165500.63,539.09,0.421,23.692,33.435,14.779,Cotton(lint),18.656


In [50]:
X_yield = yield_df.drop(["yield", "production"], axis=1)

y_yield = yield_df["yield"]

In [60]:
categorical_cols = X_yield.select_dtypes(include="object").columns.tolist()

numerical_cols = X_yield.select_dtypes(exclude="object").columns.tolist()

print(categorical_cols)
print(numerical_cols)

['season', 'state', 'recommended_crop']
['crop_year', 'area', 'annual_rainfall', 'fertilizer', 'pesticide', 'avg_temperature', 'max_temperature', 'min_temperature', 'temp_range']


In [52]:
print(categorical_cols)

['season', 'state', 'recommended_crop']


In [53]:
print(numerical_cols)

['crop_year', 'area', 'annual_rainfall', 'fertilizer', 'pesticide', 'avg_temperature', 'max_temperature', 'min_temperature', 'temp_range']


In [54]:
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import OneHotEncoder
from sklearn.ensemble import RandomForestRegressor
from sklearn.model_selection import train_test_split
import joblib
import os

# Train-Test Split
X_train, X_test, y_train, y_test = train_test_split(
    X_yield,
    y_yield,
    test_size=0.2,
    random_state=42
)

# Preprocessor
preprocessor = ColumnTransformer(
    transformers=[
        (
            "cat",
            OneHotEncoder(handle_unknown="ignore"),
            categorical_cols
        )
    ],
    remainder="passthrough"
)

# Complete Pipeline
yield_pipeline = Pipeline([
    ("preprocessor", preprocessor),
    ("model", RandomForestRegressor(
        n_estimators=200,
        random_state=42
    ))
])

# Train
yield_pipeline.fit(X_train, y_train)

# Save
os.makedirs("models", exist_ok=True)
joblib.dump(yield_pipeline, "models/yield_pipeline.pkl")

print("✅ Yield Pipeline trained and saved successfully!")

✅ Yield Pipeline trained and saved successfully!


In [55]:
model = joblib.load("models/yield_pipeline.pkl")

print(type(model))
print(type(model.named_steps["model"]))

<class 'sklearn.pipeline.Pipeline'>
<class 'sklearn.ensemble._forest.RandomForestRegressor'>


In [56]:
print(model.feature_names_in_)

['crop_year' 'season' 'state' 'area' 'annual_rainfall' 'fertilizer'
 'pesticide' 'avg_temperature' 'max_temperature' 'min_temperature'
 'recommended_crop' 'temp_range']


In [57]:
print(X_yield.columns.tolist())

['crop_year', 'season', 'state', 'area', 'annual_rainfall', 'fertilizer', 'pesticide', 'avg_temperature', 'max_temperature', 'min_temperature', 'recommended_crop', 'temp_range']


In [58]:
yield_input = {
    "crop_year": 2025,
    "season": "Kharif",
    "state": "Assam",
    "area": 5.0,
    "annual_rainfall": 1800,
    "fertilizer": 450,
    "pesticide": 60,
    "avg_temperature": 27,
    "max_temperature": 34,
    "min_temperature": 22,
    "recommended_crop": recommended_crop
}

In [59]:
yield_df_new = pd.DataFrame([yield_input])

yield_df_new["temp_range"] = (
    yield_df_new["max_temperature"] -
    yield_df_new["min_temperature"]
)

predicted_yield = yield_pipeline.predict(yield_df_new)[0]

print(f"Predicted Yield: {predicted_yield:.2f}")

Predicted Yield: 1.08


In [69]:
fert_df = pd.read_csv(r"D:\AgriFlow-AI\data\fertilizer_recommendation.csv")

In [70]:
fert_df.columns = fert_df.columns.str.lower()

fert_df["npk_total"] = (
    fert_df["nitrogen_level"] +
    fert_df["phosphorus_level"] +
    fert_df["potassium_level"]
)

fert_df["temp_humidity"] = (
    fert_df["temperature"] *
    fert_df["humidity"]
)

In [71]:
X_fert = fert_df.drop("recommended_fertilizer", axis=1)

y_fert = fert_df["recommended_fertilizer"]

In [72]:
categorical_cols = X_fert.select_dtypes(include="object").columns.tolist()

numerical_cols = X_fert.select_dtypes(exclude="object").columns.tolist()

print(categorical_cols)

print(numerical_cols)

['soil_type', 'crop_type', 'crop_growth_stage', 'season', 'irrigation_type', 'previous_crop', 'region']
['soil_ph', 'soil_moisture', 'organic_carbon', 'electrical_conductivity', 'nitrogen_level', 'phosphorus_level', 'potassium_level', 'temperature', 'humidity', 'rainfall', 'fertilizer_used_last_season', 'yield_last_season', 'npk_total', 'temp_humidity']


In [73]:
from sklearn.model_selection import train_test_split
from sklearn.compose import ColumnTransformer
from sklearn.preprocessing import OneHotEncoder
from sklearn.pipeline import Pipeline
from sklearn.ensemble import RandomForestClassifier
import joblib
import os

# Train-Test Split
X_train, X_test, y_train, y_test = train_test_split(
    X_fert,
    y_fert,
    test_size=0.2,
    random_state=42,
    stratify=y_fert
)

# Preprocessor
preprocessor = ColumnTransformer(
    transformers=[
        (
            "cat",
            OneHotEncoder(handle_unknown="ignore"),
            categorical_cols
        )
    ],
    remainder="passthrough"
)

# Complete Pipeline
fert_pipeline = Pipeline([
    ("preprocessor", preprocessor),
    ("model", RandomForestClassifier(
        n_estimators=200,
        random_state=42
    ))
])

# Train
fert_pipeline.fit(X_train, y_train)

# Save
os.makedirs("models", exist_ok=True)
joblib.dump(fert_pipeline, "models/fert_pipeline.pkl")

print("✅ Fertilizer Pipeline trained and saved successfully!")

✅ Fertilizer Pipeline trained and saved successfully!


In [74]:
model = joblib.load("models/fert_pipeline.pkl")

print(type(model))
print(type(model.named_steps["model"]))

<class 'sklearn.pipeline.Pipeline'>
<class 'sklearn.ensemble._forest.RandomForestClassifier'>


In [75]:
fert_input = {
    "soil_type": "Loamy",
    "crop_type": recommended_crop,          # Stage 1 output
    "crop_growth_stage": "Vegetative",
    "season": "Kharif",
    "irrigation_type": "Drip",
    "previous_crop": "Wheat",
    "region": "South",

    "soil_ph": 6.5,
    "soil_moisture": 35.0,
    "organic_carbon": 0.82,
    "electrical_conductivity": 0.45,

    "nitrogen_level": 90,
    "phosphorus_level": 42,
    "potassium_level": 43,

    "temperature": 27.0,
    "humidity": 82.0,
    "rainfall": 1800.0,

    "fertilizer_used_last_season": 320.0,

    "yield_last_season": predicted_yield
}

In [76]:
fert_df_new = pd.DataFrame([fert_input])

fert_df_new["npk_total"] = (
    fert_df_new["nitrogen_level"] +
    fert_df_new["phosphorus_level"] +
    fert_df_new["potassium_level"]
)

fert_df_new["temp_humidity"] = (
    fert_df_new["temperature"] *
    fert_df_new["humidity"]
)

In [77]:
recommended_fertilizer = fert_pipeline.predict(fert_df_new)[0]

print("Recommended Fertilizer :", recommended_fertilizer)

Recommended Fertilizer : NPK


In [78]:
print("=" * 65)
print("🌾 AGRIFLOW AI - CASCADED DECISION SUPPORT SYSTEM")
print("=" * 65)

print(f"🌱 Recommended Crop      : {recommended_crop}")
print(f"🌾 Predicted Yield       : {predicted_yield:.2f}")
print(f"🌿 Recommended Fertilizer: {recommended_fertilizer}")

print("=" * 65)
print("Pipeline Completed Successfully")
print("=" * 65)

🌾 AGRIFLOW AI - CASCADED DECISION SUPPORT SYSTEM
🌱 Recommended Crop      : rice
🌾 Predicted Yield       : 1.08
🌿 Recommended Fertilizer: NPK
Pipeline Completed Successfully
